In [1]:
"""
Persiapan dataset penyakit kulit (6 kelas): kumpulkan -> filter -> deduplikasi -> split.

  1. Cari folder sumber yang cocok dengan tiap kelas (folder berlabel campuran dibuang).
  2. Buang file korup dan duplikat SECARA GLOBAL (lintas kelas & sumber):
       - near-duplicate = jarak Hamming pHash DAN dHash sama-sama <= ambang
       - duplikat dalam satu kelas    -> simpan satu
       - gambar sama di beberapa kelas -> buang semua salinan (label konflik)
  3. Cap per kelas, lalu split train/val/test (stratified, seed tetap).

Output: dataset_split/, manifest.csv (jejak tiap file), dedup_report.csv.
"""

import csv
import os
import random
import shutil
import subprocess
import sys
from collections import defaultdict
from pathlib import Path

try:
    import imagehash
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "imagehash", "-q"], check=True)
    import imagehash
import numpy as np
from PIL import Image

# ===== Konfigurasi =====
SEED = 42
INPUT_ROOT = Path(os.environ.get("SKIN_INPUT_ROOT", "/kaggle/input"))
WORK_ROOT = Path(os.environ.get("SKIN_WORK_ROOT", "/kaggle/working"))
SPLIT_ROOT = WORK_ROOT / "dataset_split"
MANIFEST_CSV = WORK_ROOT / "manifest.csv"
DEDUP_REPORT_CSV = WORK_ROOT / "dedup_report.csv"

SPLIT_RATIO = {"train": 0.8, "val": 0.1}  # sisanya = test
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

DUP_HAMMING_THRESHOLD = 4      # 0 = identik saja; 4-6 = tangkap resize/kompresi/crop ringan
MAX_IMAGES_PER_CLASS = 1500    # None = tanpa cap
MAX_IMAGES_OVERRIDE = {}       # contoh: {"Urticaria": None}

EXCLUDE_MIXED_FOLDERS = True
MIXED_FOLDER_KEYWORDS = [
    "and rosacea", "lichen planus", "basal cell carcinoma",
    "malignant lesions", "and related diseases", "and other",
]
EXCLUDED_FOLDER_PATHS = ["Atopic Dermatitis Photos/Acne and Rosacea Photos"]

# Urutan dict = prioritas jika satu folder cocok lebih dari satu kelas.
TARGET_CLASSES = {
    "Acne": ["acne"],
    "Keratosis": ["keratosis"],
    "Atopic Dermatitis": ["atopic dermatitis", "atopic eczema", "dermatitis atopik", "eksim atopik"],
    "Psoriasis": ["psoriasis"],
    "Urticaria": ["urticaria", "hives"],
    "Eczema Herpeticum": ["eczema herpeticum", "herpeticum", "kaposi varicelliform"],
}

# Kata generik berisiko ("eczema" juga cocok dengan eksim biasa). True = tetap
# dipakai sebagai cadangan, tetapi folder yang cocok hanya lewat kata ini diberi peringatan.
ALLOW_GENERIC_KEYWORDS = True
GENERIC_KEYWORDS = {"Eczema Herpeticum": ["eczema"]}

CLASS_ORDER = {cls: i for i, cls in enumerate(TARGET_CLASSES)}
POPCOUNT8 = np.array([bin(i).count("1") for i in range(256)], dtype=np.uint8)


# ===== Step 1: cari folder sumber =====

def source_label(folder: Path) -> str:
    parts = folder.relative_to(INPUT_ROOT).parts
    return f"{parts[1]}/{parts[2]}" if parts[0] == "datasets" and len(parts) >= 3 else parts[0]


def find_class_folders() -> dict:
    found, generic_only = defaultdict(list), []

    for d in sorted(p for p in INPUT_ROOT.rglob("*") if p.is_dir()):
        if any(pat in str(d) for pat in EXCLUDED_FOLDER_PATHS):
            continue
        if not any(f.suffix.lower() in IMG_EXT for f in d.iterdir() if f.is_file()):
            continue

        name = d.name.lower()
        for cls, keywords in TARGET_CLASSES.items():
            generic = GENERIC_KEYWORDS.get(cls, []) if ALLOW_GENERIC_KEYWORDS else []
            hit_specific = any(k in name for k in keywords)
            hit_generic = any(k in name for k in generic)
            if hit_specific or hit_generic:
                found[cls].append(d)
                if hit_generic and not hit_specific:
                    generic_only.append((cls, d))
                break

    if EXCLUDE_MIXED_FOLDERS:
        for cls, folders in list(found.items()):
            pure = [f for f in folders if not any(k in f.name.lower() for k in MIXED_FOLDER_KEYWORDS)]
            if pure:
                found[cls] = pure
            else:
                print(f"[!] {cls}: tidak ada folder PURE, folder campuran tetap dipakai.")

    if generic_only:
        print("\n[!] PERINGATAN LABEL: folder di bawah cocok hanya lewat kata generik.")
        print("    Pastikan isinya memang kelas tsb (eksim umum BUKAN eczema herpeticum).")
        for cls, d in generic_only:
            print(f"    [{cls}] {d}")

    print("\nFolder terpakai:")
    for cls in TARGET_CLASSES:
        if not found.get(cls):
            print(f"  [!] {cls}: TIDAK ADA folder yang cocok")
        for f in found.get(cls, []):
            n = sum(1 for x in f.iterdir() if x.suffix.lower() in IMG_EXT)
            print(f"  {cls:<18} {n:>6} gambar  {f}")
    return found


# ===== Step 2: hash + deduplikasi =====

def hash_image(path: Path):
    """(pHash, dHash) sebagai integer 64-bit, atau None jika file korup."""
    try:
        with Image.open(path) as im:
            rgb = im.convert("RGB")  # memaksa decode penuh
            return tuple(int("".join("1" if b else "0" for b in h.hash.flatten()), 2)
                         for h in (imagehash.phash(rgb), imagehash.dhash(rgb)))
    except Exception:
        return None


def load_records(class_folders: dict):
    records, report = [], []
    for cls, folders in class_folders.items():
        for folder in folders:
            src = source_label(folder)
            for path in sorted(folder.iterdir()):
                if path.suffix.lower() not in IMG_EXT:
                    continue
                hashes = hash_image(path)
                if hashes is None:
                    report.append([str(path), cls, "korup", ""])
                else:
                    records.append({"path": path, "cls": cls, "src": src, "hp": hashes[0], "hd": hashes[1]})
    return records, report


def hamming(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    """Jarak Hamming semua pasangan a(m,) x b(n,) uint64 -> (m, n)."""
    x = np.ascontiguousarray(a[:, None] ^ b[None, :])
    return POPCOUNT8[x.view(np.uint8).reshape(*x.shape, 8)].sum(axis=-1)


def deduplicate(records: list, report: list) -> list:
    n = len(records)
    hp = np.array([r["hp"] for r in records], dtype=np.uint64)
    hd = np.array([r["hd"] for r in records], dtype=np.uint64)

    parent = list(range(n))

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    for start in range(0, n, 512):
        block = slice(start, start + 512)
        dist = np.maximum(hamming(hp[block], hp), hamming(hd[block], hd))
        for r, c in zip(*np.nonzero(dist <= DUP_HAMMING_THRESHOLD)):
            i = start + int(r)
            if c > i:
                parent[find(int(c))] = find(i)

    clusters = defaultdict(list)
    for i in range(n):
        clusters[find(i)].append(i)

    removed = set()
    for members in clusters.values():
        if len(members) == 1:
            continue
        members.sort(key=lambda i: (CLASS_ORDER[records[i]["cls"]], str(records[i]["path"])))
        classes = {records[i]["cls"] for i in members}

        if len(classes) > 1:  # label konflik -> buang semua salinan
            for i in members:
                removed.add(i)
                report.append([str(records[i]["path"]), records[i]["cls"],
                               "konflik_label", "|".join(sorted(classes))])
        else:                 # duplikat biasa -> simpan yang pertama
            keeper = records[members[0]]["path"]
            for i in members[1:]:
                removed.add(i)
                report.append([str(records[i]["path"]), records[i]["cls"],
                               "duplikat", f"dipertahankan:{keeper}"])

    return [r for i, r in enumerate(records) if i not in removed]


# ===== Step 3: cap + split =====

def build_split(records: list) -> list:
    if SPLIT_ROOT.exists():
        shutil.rmtree(SPLIT_ROOT)

    by_class = defaultdict(list)
    for r in records:
        by_class[r["cls"]].append(r)

    manifest = []
    print(f"\n{'Kelas':<20}{'Bersih':>8}{'Train':>8}{'Val':>8}{'Test':>8}")
    for cls in TARGET_CLASSES:
        items = sorted(by_class[cls], key=lambda r: str(r["path"]))
        if not items:
            print(f"{cls:<20}  [!] kosong, dilewati")
            continue

        rng = random.Random(f"{SEED}-{cls}")
        n_clean = len(items)
        cap = MAX_IMAGES_OVERRIDE.get(cls, MAX_IMAGES_PER_CLASS)
        if cap is not None and n_clean > cap:
            items = rng.sample(items, cap)
        rng.shuffle(items)

        n_train = int(len(items) * SPLIT_RATIO["train"])
        n_val = int(len(items) * SPLIT_RATIO["val"])
        parts = {"train": items[:n_train], "val": items[n_train:n_train + n_val],
                 "test": items[n_train + n_val:]}

        idx = 0
        for split, part in parts.items():
            out_dir = SPLIT_ROOT / split / cls
            out_dir.mkdir(parents=True)
            for r in part:
                idx += 1
                name = f"{cls.replace(' ', '_')}_{idx:04d}{r['path'].suffix.lower()}"
                shutil.copy(r["path"], out_dir / name)
                manifest.append([split, cls, name, r["src"], str(r["path"])])

        print(f"{cls:<20}{n_clean:>8}{len(parts['train']):>8}{len(parts['val']):>8}{len(parts['test']):>8}")
    return manifest


def write_csv(path: Path, header: list, rows: list):
    with open(path, "w", newline="", encoding="utf-8") as fh:
        writer = csv.writer(fh)
        writer.writerow(header)
        writer.writerows(rows)


# ===== Main =====

def main():
    random.seed(SEED)

    class_folders = find_class_folders()
    records, report = load_records(class_folders)
    print(f"\nGambar valid: {len(records)} | korup: {len(report)}")

    clean = deduplicate(records, report)
    n_dup = sum(1 for r in report if r[2] == "duplikat")
    n_conflict = sum(1 for r in report if r[2] == "konflik_label")
    print(f"Dibuang: {n_dup} duplikat, {n_conflict} konflik-label -> sisa {len(clean)} gambar")

    manifest = build_split(clean)

    write_csv(DEDUP_REPORT_CSV, ["file", "kelas", "alasan", "keterangan"], report)
    write_csv(MANIFEST_CSV, ["split", "kelas", "file", "sumber", "path_asli"], manifest)
    print(f"\nSelesai. Dataset: {SPLIT_ROOT}\nLaporan: {DEDUP_REPORT_CSV}\nManifest: {MANIFEST_CSV}")


if __name__ == "__main__":
    main()


[!] PERINGATAN LABEL: folder di bawah cocok hanya lewat kata generik.
    Pastikan isinya memang kelas tsb (eksim umum BUKAN eczema herpeticum).
    [Eczema Herpeticum] /kaggle/input/datasets/ismailpromus/skin-diseases-image-dataset/IMG_CLASSES/1. Eczema 1677
    [Eczema Herpeticum] /kaggle/input/datasets/longngzzz/sd-198/Acute_Eczema
    [Eczema Herpeticum] /kaggle/input/datasets/longngzzz/sd-198/Dry_Skin_Eczema
    [Eczema Herpeticum] /kaggle/input/datasets/longngzzz/sd-198/Dyshidrosiform_Eczema
    [Eczema Herpeticum] /kaggle/input/datasets/longngzzz/sd-198/Eczema
    [Eczema Herpeticum] /kaggle/input/datasets/longngzzz/sd-198/Nummular_Eczema
    [Eczema Herpeticum] /kaggle/input/datasets/shubhamgoel27/dermnet/test/Eczema Photos
    [Eczema Herpeticum] /kaggle/input/datasets/shubhamgoel27/dermnet/train/Eczema Photos

Folder terpakai:
  Acne                 2060 gambar  /kaggle/input/datasets/ahmedismaiil/skin-issues-version-2-dataset-balanced/Skin v2/acne
  Acne                   4